# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print summary
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields using @id references
record_sets = getattr(meta, 'recordSet', [])  # this will be a list

print('Available Record Sets:')
for recset in record_sets:
    # Each recset is a metadata object
    print(f"- @id: {recset['@id']}, Name: {recset.get('name', '<no name>')}")
    fields = recset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    - @id: {field['@id']}, Name: {field.get('name', '<no name>')}, DataType: {field.get('dataType', '<no datatype>')}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect record set @id values for extraction
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
# Loop through each record set and extract its records
for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[rec_id] = df
            print(f"Loaded {len(df)} records from record set: {rec_id}")
        else:
            print(f"No records found for record set: {rec_id}")
    except Exception as e:
        print(f"Error loading records for {rec_id}: {e}")

if dataframes:
    # Use the first non-empty record set as example
    main_recset_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set {main_recset_id}:")
    print(dataframes[main_recset_id].columns.tolist())
    display(dataframes[main_recset_id].head())
else:
    main_recset_id = None
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: filter by a numeric field, normalize, group by category

if main_recset_id is not None:
    df = dataframes[main_recset_id]
    # Choose a likely numeric field using field @id (search for common protein dataset columns)
    # This will depend on schema: Substitute with a real @id for 'molecular_weight' or similar numeric column.
    numeric_field_candidates = [
        col for col in df.columns if 'weight' in col.lower() or 'count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower() or 'coverage' in col.lower()
    ]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        # Default to the first field
        numeric_field_id = df.columns[0]
        print(f"Using default field: {numeric_field_id}")

    # Ensure we handle missing or non-numeric data
    df_clean = df.copy()
    df_clean[numeric_field_id] = pd.to_numeric(df_clean[numeric_field_id], errors='coerce')

    threshold = df_clean[numeric_field_id].dropna().median()  # Use median as demonstration
    filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by a categorical field (e.g., one with 'sample', 'group', or 'description' in its name)
    group_field_candidates = [
        col for col in df.columns if any([k in col.lower() for k in ['sample', 'group', 'condition', 'desc']])
    ]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field}:")
        display(grouped_df.head())
    else:
        print("No obvious group field found for aggregation.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_recset_id is not None and numeric_field_id in df_clean.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df_clean[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If we found a group_field, visualize mean by group
    if 'group_field' in locals() and group_field in df_clean.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df_clean)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The Mass Spectrometry dataset provides rich information on protein abundance, modifications, and sequence analysis from human extracellular vesicles.
- We loaded the Croissant-standardized dataset using the `mlcroissant` library, programmatically explored available record sets and fields, and extracted data using their unique `@id`s.
- Preliminary EDA reported the distribution and basic statistics of a key numeric field, with filtering and normalization, and visualized distributions by group where possible.
- For robust workflows, always verify field meanings, datatypes, and refer to schema documentation to guide custom analyses.